<a href="https://colab.research.google.com/github/acnoori/EE434/blob/main/SmartGrid2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab Exercises for Session 2: Smart Grid Architecture, Communication & IEC 61850 Substation Automation

## Lab Title: Simulation of Substation Automation & Cyber-Physical Communication Dynamics: IEC 61850 GOOSE Messaging vs. Legacy DNP3 SCADA Protection & Control

**Course**: Intelligent Electric Energy Networks (شبکه‌های هوشمند انرژی الکتریکی)

**Target Level**: M.Sc. / Ph.D. Level Electrical Engineering

## 1. Lab Overview & Objectives

In Session 2 (معماری و زیرساخت شبکه هوشمند), we analyzed the NIST 4.0 Framework, the TSO-DSO-Aggregator Control Hierarchy, and the Application Protocols governing modern smart substations.

### Learning Objectives:

*   **Model Substation Protection Latency**: Compare peer-to-peer IEC 61850 GOOSE (Generic Object Oriented Substation Events) multicast messaging against master-slave DNP3 / SCADA polling over IP/Ethernet.
*   **Simulate Closed-Loop Cyber-Physical Response**: Observe how network latency and cyber communication delays directly impact the physical fault clearing time and thermal/voltage stress on substation transformers and lines.
*   **Analyze Communication Reliability & Packet Loss**: Evaluate the impact of packet drops on protection reliability and circuit breaker actuation.

## 2. Python Implementation (lab2_substation_automation.py)

This script models a short-circuit fault event on a MV distribution substation feeder and compares the trip response time between IEC 61850 GOOSE multicast messaging ($< 3\text{ ms}$ network delay) and Legacy DNP3 SCADA polling ($45\text{ ms}$ polling delay).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ==============================================================================
# LAB 2: IEC 61850 GOOSE MESSAGING & CYBER-PHYSICAL COMMUNICATION LATENCY
# Course: Intelligent Electric Energy Networks (Smart Grids)
# Session 2: Smart Grid Architecture, Communication & Interoperability
# Scenario: MV Substation Feeder Short-Circuit Fault Protection
# ==============================================================================

# ------------------------------------------------------------------------------
# 1. SIMULATION SETUP & TIME HORIZON
# ------------------------------------------------------------------------------
t_end_ms = 100.0                       # Simulation window: 100 milliseconds
dt_ms = 0.1                            # Resolution: 0.1 ms (100 us timestep)
time_ms = np.arange(0, t_end_ms, dt_ms)

# Fault Event Parameters
t_fault_ms = 10.0                      # Over-current fault occurs at t = 10 ms
fault_current_threshold_ka = 2.0       # IED Over-current Pickup Threshold (kA)
fault_current_peak_ka = 8.5            # Short-circuit Peak Current (kA)

# Generate AC Short-Circuit Current Waveform with Decaying DC Offset (50 Hz)
freq_hz = 50.0
omega = 2 * np.pi * (freq_hz / 1000.0) # omega in rad/ms
current_ka = np.zeros_like(time_ms)
for i, t in enumerate(time_ms):
    if t >= t_fault_ms:
        dt_f = t - t_fault_ms
        current_ka[i] = fault_current_peak_ka * (np.sin(omega * dt_f) + np.exp(-dt_f / 15.0))
    else:
        current_ka[i] = 0.4 * np.sin(omega * t)

# ------------------------------------------------------------------------------
# 2. COMMUNICATION PROTOCOL LATENCY MODELS
# ------------------------------------------------------------------------------
# IEC 61850 GOOSE (Multicast publisher/subscriber over Ethernet MAC layer)
goose_net_delay_ms = 3.0              # Network propagation + IED parsing delay
breaker_mech_time_ms = 15.0           # Circuit breaker mechanical opening time

# Legacy DNP3 / SCADA Polling (Master-Slave polling over TCP/IP)
dnp3_polling_delay_ms = 45.0          # Polling interval + serial/IP network latency

# ------------------------------------------------------------------------------
# 3. CLOSED-LOOP PROTECTION RESPONSE SIMULATION
# ------------------------------------------------------------------------------
# Case A: IEC 61850 GOOSE Protection Response
ied_detect_goose = False
t_trip_goose = None
current_goose_ka = np.copy(current_ka)

for i, t in enumerate(time_ms):
    if t >= t_fault_ms and abs(current_goose_ka[i]) >= fault_current_threshold_ka and not ied_detect_goose:
        ied_detect_goose = True
        t_trip_goose = t + goose_net_delay_ms + breaker_mech_time_ms

    if t_trip_goose is not None and t >= t_trip_goose:
        current_goose_ka[i] = 0.0 # Circuit breaker opens and clears fault

# Case B: Legacy DNP3 / SCADA Polling Protection Response
ied_detect_dnp3 = False
t_trip_dnp3 = None
current_dnp3_ka = np.copy(current_ka)

for i, t in enumerate(time_ms):
    if t >= t_fault_ms and abs(current_dnp3_ka[i]) >= fault_current_threshold_ka and not ied_detect_dnp3:
        ied_detect_dnp3 = True
        t_trip_dnp3 = t + dnp3_polling_delay_ms + breaker_mech_time_ms

    if t_trip_dnp3 is not None and t >= t_trip_dnp3:
        current_dnp3_ka[i] = 0.0 # Circuit breaker opens and clears fault

# ------------------------------------------------------------------------------
# 4. RESULTS PRINT & VISUALIZATION
# ------------------------------------------------------------------------------
print("--- LAB 2 SIMULATION RESULTS ---")
print(f"Fault Occurrence Time       : {t_fault_ms:.2f} ms")
print(f"IEC 61850 GOOSE Fault Clear : {t_trip_goose:.2f} ms (Total Cleared Time: {t_trip_goose - t_fault_ms:.2f} ms)")
print(f"Legacy DNP3 SCADA Clear    : {t_trip_dnp3:.2f} ms (Total Cleared Time: {t_trip_dnp3 - t_fault_ms:.2f} ms)")

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

# Subplot 1: IEC 61850 GOOSE Response
ax1.plot(time_ms, current_goose_ka, 'r-', label='Fault Current (IEC 61850 GOOSE Cleared)', linewidth=2)
ax1.axvline(t_fault_ms, color='k', linestyle='--', label='Fault Inception (10 ms)')
ax1.axvline(t_trip_goose, color='g', linestyle='-.', label=f'GOOSE Breaker Open ({t_trip_goose:.1f} ms)')
ax1.axhline(fault_current_threshold_ka, color='orange', linestyle=':', label='IED Threshold (2 kA)')
ax1.set_ylabel('Current (kA)', fontsize=11, fontweight='bold')
ax1.set_title('Lab 2: Substation Automation - IEC 61850 GOOSE vs Legacy SCADA Protection', fontsize=12, fontweight='bold')
ax1.grid(True, linestyle='--', alpha=0.5)
ax1.legend(loc='upper right', fontsize=9)

# Subplot 2: Legacy DNP3 SCADA Response
ax2.plot(time_ms, current_dnp3_ka, 'b-', label='Fault Current (Legacy DNP3 Cleared)', linewidth=2)
ax2.axvline(t_fault_ms, color='k', linestyle='--')
ax2.axvline(t_trip_dnp3, color='purple', linestyle='-.', label=f'DNP3 Breaker Open ({t_trip_dnp3:.1f} ms)')
ax2.axhline(fault_current_threshold_ka, color='orange', linestyle=':')
ax2.set_xlabel('Time (milliseconds)', fontsize=11, fontweight='bold')
ax2.set_ylabel('Current (kA)', fontsize=11, fontweight='bold')
ax2.grid(True, linestyle='--', alpha=0.5)
ax2.legend(loc='upper right', fontsize=9)

plt.tight_layout()
plt.show()

## 3. MATLAB Implementation (lab2_substation_automation.m)

This equivalent MATLAB script is compatible with MATLAB R2020b+ and GNU Octave.

```matlab
%% LAB 2: IEC 61850 GOOSE MESSAGING & SUBSTATION AUTOMATION SIMULATION
% Course: Intelligent Electric Energy Networks (Smart Grids)
% Session 2: Smart Grid Architecture, Communication & Interoperability

clear; clc; close all;

%% 1. SIMULATION PARAMETERS
t_end_ms = 100.0;                       % Total simulation time in ms
dt_ms = 0.1;                            % Timestep (0.1 ms)
time_ms = 0:dt_ms:(t_end_ms - dt_ms);

t_fault_ms = 10.0;                      % Fault inception time (ms)
fault_threshold_ka = 2.0;               % Over-current pickup threshold (kA)
fault_peak_ka = 8.5;                    % Peak short-circuit current (kA)

% Generate 50 Hz Short-Circuit Current with Decaying DC Offset
freq_hz = 50.0;
omega = 2 * pi * (freq_hz / 1000.0);    % rad/ms
current_ka = zeros(size(time_ms));

for i = 1:length(time_ms)
    t = time_ms(i);
    if t >= t_fault_ms
        dt_f = t - t_fault_ms;
        current_ka(i) = fault_peak_ka * (sin(omega * dt_f) + exp(-dt_f / 15.0));
    else
        current_ka(i) = 0.4 * sin(omega * t);
    end
end

%% 2. COMMUNICATION PROTOCOL DELAYS
goose_net_delay = 3.0;                  % IEC 61850 GOOSE delay (ms)
breaker_mech_time = 15.0;               % Circuit breaker opening time (ms)
dnp3_polling_delay = 45.0;              % Legacy DNP3 SCADA polling delay (ms)

%% 3. CLOSED-LOOP PROTECTION RESPONSE
% Case A: IEC 61850 GOOSE Response
current_goose_ka = current_ka;
ied_detect_goose = false;
t_trip_goose = NaN;

for i = 1:length(time_ms)
    t = time_ms(i);
    if t >= t_fault_ms && abs(current_goose_ka(i)) >= fault_threshold_ka && ~ied_detect_goose
        ied_detect_goose = true;
        t_trip_goose = t + goose_net_delay + breaker_mech_time;
    end
    if ~isnan(t_trip_goose) && t >= t_trip_goose
        current_goose_ka(i) = 0.0; % Breaker opens
    end
end

% Case B: Legacy DNP3 SCADA Response
current_dnp3_ka = current_ka;
ied_detect_dnp3 = false;
t_trip_dnp3 = NaN;

for i = 1:length(time_ms)
    t = time_ms(i);
    if t >= t_fault_ms && abs(current_dnp3_ka(i)) >= fault_threshold_ka && ~ied_detect_dnp3
        ied_detect_dnp3 = true;
        t_trip_dnp3 = t + dnp3_polling_delay + breaker_mech_time;
    end
    if ~isnan(t_trip_dnp3) && t >= t_trip_dnp3
        current_dnp3_ka(i) = 0.0; % Breaker opens
    end
end

%% 4. RESULTS DISPLAY & VISUALIZATION
fprintf('--- LAB 2 SIMULATION RESULTS ---\n');
fprintf('Fault Inception Time       : %.2f ms\n', t_fault_ms);
fprintf('IEC 61850 GOOSE Clearing   : %.2f ms (Delay: %.2f ms)\n', t_trip_goose, t_trip_goose - t_fault_ms);
fprintf('Legacy DNP3 SCADA Clearing : %.2f ms (Delay: %.2f ms)\n', t_trip_dnp3, t_trip_dnp3 - t_fault_ms);

figure('Position', [100, 100, 900, 600]);

% Subplot 1: GOOSE Response
subplot(2,1,1);
plot(time_ms, current_goose_ka, 'r-', 'LineWidth', 2.0); hold on;
xline(t_fault_ms, '--k', 'Fault Inception (10 ms)');
xline(t_trip_goose, '-.g', sprintf('GOOSE Clear (%.1f ms)', t_trip_goose), 'LineWidth', 1.5);
yline(fault_threshold_ka, ':o', 'IED Threshold (2 kA)');
ylabel('Current (kA)', 'FontSize', 11, 'FontWeight', 'bold');
title('Lab 2: Substation Automation - IEC 61850 GOOSE vs Legacy SCADA Protection', 'FontSize', 12, 'FontWeight', 'bold');
grid on;

% Subplot 2: DNP3 Response
subplot(2,1,2);
plot(time_ms, current_dnp3_ka, 'b-', 'LineWidth', 2.0); hold on;
xline(t_fault_ms, '--k');
xline(t_trip_dnp3, '-.m', sprintf('DNP3 Clear (%.1f ms)', t_trip_dnp3), 'LineWidth', 1.5);
yline(fault_threshold_ka, ':o');
xlabel('Time (milliseconds)', 'FontSize', 11, 'FontWeight', 'bold');
ylabel('Current (kA)', 'FontSize', 11, 'FontWeight', 'bold');
grid on;
```

## 4. Student Lab Assignment & Analytical Questions


*   **Mapping NIST 4.0 Domains**: Map each physical component in this lab (Feeder CT/PT Sensor, Protection IED, Substation Switch, Circuit Breaker) into its corresponding NIST 4.0 Domain (Distribution, Operations, or Customer/DER) and identify the primary communication interface.
*   **Thermal Stress Energy Calculation ($I^2 t$)**: Calculate the total thermal energy stress $I^2 t$ ($\text{kA}^2 \cdot \text{s}$) absorbed by the distribution transformer under IEC 61850 GOOSE protection ($18\text{ ms}$ clearing) versus Legacy DNP3 SCADA protection ($60\text{ ms}$ clearing). Evaluate how GOOSE messaging protects physical transformer insulation against thermal degradation.
*   **Impact of Cyber Packet Drops**: IEC 61850 GOOSE utilizes re-transmission mechanisms ($T_0, T_1, T_2$) over UDP/Ethernet without TCP re-transmissions to maintain low latency. Modify the Python script to simulate a $20\%$ packet drop rate on the substation switch. How does packet loss affect the trip time and reliability of GOOSE protection?